# Linear regression with scikit-learn

You will work again to yesterday's dataset and change the objective : you will predict to predict the Salary as a function of the other variables.

To do so, you will train a linear regression model and evaluate its performances. You will see that the preprocessings are quite similar to the ones from yesterday's template (except that the label encoding step is skipped here, because our label is already a numeric variable !). The dataset contains HR records from a European company — each row is one employee, with their country, department, age, years of experience, education level, salary, and whether they were promoted. EDA is an important first step: it lets you understand the relationships between variables before committing to a model 🤓

## What you will learn in this course 🧐🧐

This lecture is a follow-along demo to walk you through building a linear regression model in practice using sklearn!

* EDA
* Baseline model : simple linear regression
    * Training pipeline
    * Test pipeline
* Multivariate linear regression
    * Training pipeline
    * Test pipeline


## EDA


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import  OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

import plotly.express as px
import plotly.graph_objects as go
# import plotly.io as pio
# # setting Jedha color palette as default
# pio.templates["jedha"] = go.layout.Template(
#     layout_colorway=["#4B9AC7", "#4BE8E0", "#9DD4F3", "#97FBF6", "#2A7FAF", "#23B1AB", "#0E3449", "#015955"]
# )
# pio.templates.default = "jedha"
# pio.renderers.default = "svg" # to be replaced by "iframe" if working on JULIE

In [2]:
# # Import dataset
# print("Loading dataset...")
# dataset = pd.read_csv("src/Data.csv")
# print("...Done.")
# print()

In [3]:
# # Basic stats
# data_desc = dataset.describe(include='all')
# print(dataset.shape)
# data_desc

In [4]:
from preprocess_data import preprocess
dataset = preprocess("src/Dataset_preprocessing.csv")
dataset.head()

Loading dataset...
...Done.

Dropping useless columns...
Dropping outliers in Age...
Done. Number of lines remaining :  502

Dropping outliers in Salary...
Done. Number of lines remaining :  502



,Country,Department,Age,YearsExperience,EducationLevel,Salary,Promoted
0,Germany,Marketing,NaN,15,Bachelor,70000,No
1,Italy,Sales,25.0,1,Bachelor,32000,No
2,UK,Sales,42.0,18,Master,77500,No
3,Spain,Finance,44.0,26,Master,95000,Yes
4,France,Marketing,44.0,16,Master,80000,No


- Target variable : Y = 'Salary'
- Explanatory variables : 'Country', 'Department', 'Age', 'YearsExperience', 'EducationLevel'
- Number of examples : 500

- Imputation of missing values : 'Age'
- Normalisation : 'Age', 'YearsExperience'
- OHE : 'Country', 'Department', 'EducationLevel'


In [5]:
# Univariate analysis
# Distribution of each numeric variable
num_features = ['Age', 'YearsExperience', 'Salary']
for i in range(len(num_features)):
    fig = px.histogram(dataset[num_features[i]])
    fig.show()

In [6]:
# Univariate analysis
# Barplot of each qualitative variable
cat_features = ['Country', 'Department', 'EducationLevel']
for i in range(len(cat_features)):
    fig = px.bar(dataset[cat_features[i]])
    fig.show()

In [7]:
# Correlation matrix
corr_matrix = dataset.corr().round(2)

import plotly.figure_factory as ff

fig = ff.create_annotated_heatmap(corr_matrix.values,
                                  x = corr_matrix.columns.tolist(),
                                  y = corr_matrix.index.tolist())


fig.show()

ValueError: could not convert string to float: 'Germany'

In [8]:
# Correlation matrix
corr_matrix = dataset.corr(numeric_only=True).round(2)

import plotly.figure_factory as ff

fig = ff.create_annotated_heatmap(corr_matrix.values,
                                  x = corr_matrix.columns.tolist(),
                                  y = corr_matrix.index.tolist())


fig.show()

Both `Age` and `YearsExperience` correlate strongly with `Salary`, with `YearsExperience` being slightly higher. This makes intuitive sense: salary tends to grow with time on the job. Use `YearsExperience` as the single feature for our baseline model.

Also look at pairwise scatterplots to investigate the interdependencies between the variables :


In [9]:
# Visualize pairwise dependencies
fig = px.scatter_matrix(dataset)
fig.update_layout(
        title = go.layout.Title(text = "Bivariate analysis", x = 0.5), showlegend = False, 
            autosize=False, height=800, width = 800)
fig.show()

## Baseline model : simple linear regression
Try a first basic model : simple linear regression with only one feature. We choose the age because we just noticed that it is strongly correlated to the salary.


In [10]:
# Separate target variable Y from features X
print("Separating labels from features...")
features_list = ["YearsExperience"]
target_variable = "Salary"

X = dataset.loc[:,features_list]
Y = dataset.loc[:,target_variable]

print("...Done.")
print()

print('Y : ')
print(Y.head())
print()
print('X :')
print(X.head())

Separating labels from features...
...Done.

Y : 
0    70000
1    32000
2    77500
3    95000
4    80000
Name: Salary, dtype: int64

X :
   YearsExperience
0               15
1                1
2               18
3               26
4               16


In [11]:
# Divide dataset Train set & Test set 
print("Dividing into train and test sets...")
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)
print("...Done.")
print()

Dividing into train and test sets...
...Done.



### Preprocessing

<Note type="note" title="Simplified syntax">

In this first example, we don't need to use the class `ColumnTransformer` because the preprocessing is very simple : it just consists in one step of imputation followed by a standardization, and these steps will be applied to the whole DataFrame (as there's actually only one column!). In this case, we can just create a `Pipeline` that describes the different steps and call the `fit_transform` and `transform` methods directly on it, by passing X_train/X_test as argument.

</Note>


In [12]:
print("Preprocessing X_train...")
print(X_train.head())
print()
preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
X_train = preprocessor.fit_transform(X_train)
print("...Done!")
print(X_train[0:5,:]) # X_train is now a numpy array
print() 

Preprocessing X_train...
     YearsExperience
107               13
336               35
71                 6
474               30
6                 27

...Done!
[[-0.35263514]
 [ 1.91246649]
 [-1.0733493 ]
 [ 1.39767066]
 [ 1.08879317]]



In [13]:
# Test pipeline
print("Preprocessing X_test...")
print(X_test.head())
print()
X_test = preprocessor.transform(X_test)
print("...Done!")
print(X_test[0:5,:]) # X_test is now a numpy array
print() 

Preprocessing X_test...
     YearsExperience
90                12
254               15
283                2
445               12
461               11

...Done!
[[-0.45559431]
 [-0.14671681]
 [-1.48518596]
 [-0.45559431]
 [-0.55855347]]



### Train model


In [14]:
# Train model
print("Train model...")
regressor = LinearRegression()
regressor.fit(X_train, Y_train)
print("...Done.")

Train model...
...Done.


### Performance assessment


In [15]:
# Predictions on training set
print("Predictions on training set...")
Y_train_pred = regressor.predict(X_train)
print("...Done.")
print(Y_train_pred)
print()

Predictions on training set...
...Done.
[ 69503.41345612 116428.20293504  54572.79862192 105763.47805347
  99364.64312452  58838.68857455  97231.69814821  58838.68857455
  50306.90866929  69503.41345612  82301.08331401  88699.91824295
  88699.91824295  43908.07374035  56705.74359824  60971.63355087
  43908.07374035  67370.46847981  82301.08331401  71636.35843244
  52439.85364561  69503.41345612  69503.41345612  46041.01871667
  92965.80819558  86566.97326664  67370.46847981  43908.07374035
  78035.19336138  54572.79862192  50306.90866929  63104.57852718
  73769.30340875  75902.24838507  43908.07374035  90832.86321927
  92965.80819558  56705.74359824 116428.20293504  52439.85364561
  56705.74359824  41775.12876404 116428.20293504  75902.24838507
  71636.35843244  75902.24838507  97231.69814821  73769.30340875
  84434.02829032  69503.41345612  48173.96369298  46041.01871667
  82301.08331401  48173.96369298  71636.35843244  78035.19336138
  63104.57852718  99364.64312452  52439.85364561  

In [16]:
# Predictions on test set
print("Predictions on test set...")
Y_test_pred = regressor.predict(X_test)
print("...Done.")
print(Y_test_pred)
print()

Predictions on test set...
...Done.
[ 67370.46847981  73769.30340875  46041.01871667  67370.46847981
  65237.52350349  99364.64312452  73769.30340875 110029.3680061
  63104.57852718  52439.85364561  54572.79862192  73769.30340875
  95098.7531719   43908.07374035  82301.08331401 112162.31298241
  63104.57852718  95098.7531719   78035.19336138  97231.69814821
  48173.96369298  48173.96369298  99364.64312452  95098.7531719
 101497.58810084  46041.01871667  67370.46847981  41775.12876404
  63104.57852718  52439.85364561  41775.12876404 107896.42302978
  69503.41345612  63104.57852718 103630.53307715  48173.96369298
  69503.41345612  43908.07374035 105763.47805347  58838.68857455
 114295.25795872 110029.3680061   43908.07374035  95098.7531719
  50306.90866929  43908.07374035  54572.79862192  48173.96369298
  80168.1383377   56705.74359824  80168.1383377   69503.41345612
  56705.74359824 105763.47805347  75902.24838507  86566.97326664
  54572.79862192 110029.3680061   80168.1383377   84434.0

In [17]:
# Print R^2 scores
print("R2 score on training set : ", r2_score(Y_train, Y_train_pred))
print("R2 score on test set : ", r2_score(Y_test, Y_test_pred))

R2 score on training set :  0.7986279581364362
R2 score on test set :  0.8368408586003948


In [18]:
# Visualize the model
# Visualize predictions on training Set
fig = px.scatter(x = X_train.flatten().tolist(), y = Y_train, title = "training set")
fig.add_trace(go.Scatter(x = X_train.flatten().tolist(), y = Y_train_pred, name = "linear regression"))
fig.show()

# Visualize predictions on test Set
fig = px.scatter(x = X_test.flatten().tolist(), y = Y_test, title = "test set")
fig.add_trace(go.Scatter(x = X_test.flatten().tolist(), y = Y_test_pred, name = "linear regression"))
fig.show()

In the above example all assumptions needed to train a simple linear regression model are verified.

* There seems to be a linear relation between $Y$ and the feature $X$
* The deviation between the prediction and the actual values of $Y$, i.e. the residuals, seem to be comparable accross the values of $Y$ (this is Homoscedaticity).
* There does not seem to be any autocorrelation going on with the residuals, they seem to be randomly distributed accross all values of $Y$

In addition to that, our model seems to be a good predictor based on our training samples.


## Multivariate linear regression
Train a multivariate model by adding the rest of the features: `Department`, `Age`, `EducationLevel` and `Country`.


In [19]:
# Separate target variable Y from features X
print("Separating labels from features...")
features_list = ["Country", "Department", "Age", "YearsExperience", "EducationLevel"]
target_variable = "Salary"

X = dataset.loc[:,features_list]
Y = dataset.loc[:,target_variable]

print("...Done.")
print()

print('Y : ')
print(Y.head())
print()
print('X :')
print(X.head())

Separating labels from features...
...Done.

Y : 
0    70000
1    32000
2    77500
3    95000
4    80000
Name: Salary, dtype: int64

X :
   Country Department   Age  YearsExperience EducationLevel
0  Germany  Marketing   NaN               15       Bachelor
1    Italy      Sales  25.0                1       Bachelor
2       UK      Sales  42.0               18         Master
3    Spain    Finance  44.0               26         Master
4   France  Marketing  44.0               16         Master


In [20]:
# Automatically detect names of numeric/categorical columns
numeric_features = []
categorical_features = []
for i,t in X.dtypes.items():
    if ('float' in str(t)) or ('int' in str(t)) :
        numeric_features.append(i)
    else :
        categorical_features.append(i)

print('Found numeric features ', numeric_features)
print('Found categorical features ', categorical_features)

Found numeric features  ['Age', 'YearsExperience']
Found categorical features  ['Country', 'Department', 'EducationLevel']


In [21]:
# Divide dataset Train set & Test set 
print("Dividing into train and test sets...")
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)
print("...Done.")
print()

Dividing into train and test sets...
...Done.



### Preprocessing


In [22]:
# Create pipeline for numeric features
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')), # missing values will be replaced by columns' mean
    ('scaler', StandardScaler())
])

In [23]:
# Create pipeline for categorical features
categorical_transformer = Pipeline(
    steps=[
    ('encoder', OneHotEncoder(drop='first')) # first column will be dropped to avoid creating correlations between features
    ])

In [24]:
# Use ColumnTransformer to make a preprocessor object that describes all the treatments to be done
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [25]:
# Preprocessings on train set
print("Performing preprocessings on train set...")
print(X_train.head())
X_train = preprocessor.fit_transform(X_train)
print('...Done.')
print(X_train[0:5]) # MUST use this syntax because X_train is a numpy array and not a pandas DataFrame anymore
print()

# Preprocessings on test set
print("Performing preprocessings on test set...")
print(X_test.head()) 
X_test = preprocessor.transform(X_test) # Don't fit again !! The test set is used for validating decisions
# we made based on the training set, therefore we can only apply transformations that were parametered using the training set.
# Otherwise this creates what is called a leak from the test set which will introduce a bias in all your results.
print('...Done.')
print(X_test[0:5,:]) # MUST use this syntax because X_test is a numpy array and not a pandas DataFrame anymore
print()

Performing preprocessings on train set...
    Country   Department   Age  YearsExperience EducationLevel
107      UK        Sales  34.0               13       Bachelor
336      UK           HR  62.0               35         Master
71   France      Finance  32.0                6       Bachelor
474   Spain  Engineering  50.0               30       Bachelor
6    France           HR  46.0               27            PhD
...Done.
[[-0.58144657 -0.35263514  0.          0.          0.          1.
   0.          0.          0.          1.          0.          0.        ]
 [ 2.46321912  1.91246649  0.          0.          0.          1.
   0.          1.          0.          0.          1.          0.        ]
 [-0.79892269 -1.0733493   0.          0.          0.          0.
   1.          0.          0.          0.          0.          0.        ]
 [ 1.15836239  1.39767066  0.          0.          1.          0.
   0.          0.          0.          0.          0.          0.        ]
 [ 0.72

### Train model


In [26]:
# Train model
print("Train model...")
regressor = LinearRegression()
regressor.fit(X_train, Y_train)
print("...Done.")

Train model...
...Done.


### Performance assessment


In [27]:
# Predictions on training set
print("Predictions on training set...")
Y_train_pred = regressor.predict(X_train)
print("...Done.")
print(Y_train_pred)
print()

Predictions on training set...
...Done.
[ 62277.37683175 117938.95911138  48631.92691985 102452.5432891
 108545.30502635  51002.71422133  95975.7482673   59887.55064051
  46092.45631451  65040.84096797 100108.93094369  96608.92013548
 107446.95276482  39459.20422633  46402.28984721  49764.23474978
  58411.14808358  82314.88436813  77972.50511489  75796.06793662
  56757.98992154  64436.35862909  74155.10136858  38383.6350814
  84031.55588009  76647.57392937  61967.12504071  59565.32138036
  84778.55369042  50227.74861225  40009.64725001  58407.5888798
  66606.3781011   85082.96745861  45378.16806151  88301.15211011
  89048.15872398  53854.13761049 105609.51632574  46948.88069403
  53590.61376914  33015.79907972 135610.0408087   74648.64933867
  77259.91824566  76911.78258054  93281.36366803  66115.29601378
  80046.27240579  64204.51877703  36835.82446412  53431.97190641
  74717.85198404  38130.87359437  72850.68931402  69481.62507783
  51637.23421121 102282.67967287  67432.78610361  543

In [28]:
# Predictions on test set
print("Predictions on test set...")
Y_test_pred = regressor.predict(X_test)
print("...Done.")
print(Y_test_pred)
print()

Predictions on test set...
...Done.
[ 62331.30985291  78978.1100072   44620.92896827  71840.85601554
  76805.92352147 104640.86351287  87925.14204073 106763.12127686
  58388.34553855  56920.85279203  43584.04259069  71972.81467277
 107910.78503648  30150.42589318  86708.27563575 111750.86768886
  73741.15776857 100453.6186984   77031.90060051  93281.36366803
  45536.86659714  43021.48871179 100042.59221022  94949.80983332
  91707.39424415  43069.09820787  66314.00137124  36474.0384147
  64785.17556067  42536.19021595  45086.10218549 104448.87592382
  70582.1054579   52972.78592352 118712.33298536  42688.95896164
  64041.07286707  55244.61303918 112033.2125985   57102.98008867
 124212.86154127 118120.28751476  38877.75656795  91276.60228593
  46465.98096098  32935.14927022  52692.05309436  48363.01311708
  75658.25976185  53722.37568982  79718.38593636  64514.00274796
  57229.09292869 104710.06615823  65339.87100553  78997.74616985
  44219.80641484 100786.65360604  72099.7330328   79791

In [29]:
# Print R^2 scores
print("R2 score on training set : ", r2_score(Y_train, Y_train_pred))
print("R2 score on test set : ", r2_score(Y_test, Y_test_pred))

R2 score on training set :  0.9004459810707417
R2 score on test set :  0.9096836387083778


### Conclusion about the model's quality
Here we can see that the R2-score has significantly improved when we went from simple to multiple linear regression. This is because we added some features that were indeed useful to predict the values of the target variables. 
Sometimes, adding more features won't necessarily improve the performances, and it can even deteriorate in some cases ! That's why in general, some feature selection methods are required to choose the right set of variables to be included into the model.

<Note type="note" title="What about plotting ?">
You can notice we didn't plot the multivariate model. It's simply because we can't ! To make a graphical representation of this model, we would need 4 dimensions (3 features + the target). Unfortunately, our brain is able to visualize only 3 at a time 🥴
This is the reason why it's so important to always compute some metrics such as the R2-score to assess the quality of our predictions: most of the times, we're "blind" on what's happening and we have to rely on these scores to make sure the model is making good predictions !
</Note>


### Interpreting the model's coefficients
As we've standardized our features, we can use the coefficients of the regression to estimate the importance of each feature for the prediction. The model's parameters are saved in a `.coef_` attribute:


In [30]:
regressor.coef_

array([-1211.736904  , 22477.31275798,  2567.00684007,   332.52975015,
         309.48397094,  3089.67036129,  -340.95849753, -6592.63261832,
       -2931.19165228, -5721.37257684,  9308.77833315, 19410.56670293])

Each coefficient can be linked with the name of the corresponding feature by digging into the different pipelines that were used to produce the final version of the X_train/X_test arrays:


In [31]:
column_names = []
for name, pipeline, features_list in preprocessor.transformers_: # loop over pipelines
    if name == 'num': # if pipeline is for numeric variables
        features = features_list # just get the names of columns to which it has been applied
    else: # if pipeline is for categorical variables
        features = pipeline.named_steps['encoder'].get_feature_names_out() # get output columns names from OneHotEncoder
    column_names.extend(features) # concatenate features names
        
print("Names of columns corresponding to each coefficient: ", column_names)

Names of columns corresponding to each coefficient:  ['Age', 'YearsExperience', 'Country_Germany', 'Country_Italy', 'Country_Spain', 'Country_UK', 'Department_Finance', 'Department_HR', 'Department_Marketing', 'Department_Sales', 'EducationLevel_Master', 'EducationLevel_PhD']


In [32]:
# Create a pandas DataFrame
coefs = pd.DataFrame(index = column_names, data = regressor.coef_.transpose(), columns=["coefficients"])
coefs

,coefficients
Age,-1211.736904
YearsExperience,22477.312758
Country_Germany,2567.006840
Country_Italy,332.529750
Country_Spain,309.483971
Country_UK,3089.670361
Department_Finance,-340.958498
Department_HR,-6592.632618
Department_Marketing,-2931.191652
Department_Sales,-5721.372577


<Note type="warning" title="Comparison in absolute values">
The feature importance is related to the absolute values of the coefficients. Don't forget to compute the absolute value before concluding !
</Note>


In [33]:
# Compute abs() and sort values
feature_importance = abs(coefs).sort_values(by = 'coefficients')
feature_importance

,coefficients
Country_Spain,309.483971
Country_Italy,332.529750
Department_Finance,340.958498
Age,1211.736904
Country_Germany,2567.006840
Department_Marketing,2931.191652
Country_UK,3089.670361
Department_Sales,5721.372577
Department_HR,6592.632618
EducationLevel_Master,9308.778333


In [34]:
# Plot coefficients
fig = px.bar(feature_importance, orientation = 'h')
fig.update_layout(showlegend = False, 
                  margin = {'l': 120} # to avoid cropping of column names
                 )
fig.show()